## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **0d69e825-1a18-4e42-8a2a-2afe6e013d55**


## Dataset

# Typologies des communes


## Description

Ce jeu de données présente les typologies des espaces territoriaux pour les communes fribourgeoises. Il regroupe une sélection des typologies les plus pertinentes.Agglomérations statistiques &#40;OFS&#41; — objet statistique, sans effet juridique, produit par un algorithme uniforme sur toute la Suisse. Une trame régulière mesure la densité de population, d&#x27;emplois et de nuitées hôtelières pour identifier les noyaux ; les couronnes sont déterminées par l&#x27;intensité des flux de pendulaires vers ces noyaux ; l&#x27;ensemble noyau + couronne doit atteindre 20 000 habitants pour être qualifié d&#x27;agglomération. Les codes correspondent au numéro OFS de la commune-centre.Périmètres VACo &#40;ARE&#41; — objet juridico-financier. VACo signifie « Villes et Agglomérations ayant droit aux Contributions » : ce périmètre n&#x27;a pas de vocation descriptive, il détermine quelles communes peuvent bénéficier des contributions fédérales du programme en faveur du trafic d&#x27;agglomération. Il est fixé par ordonnance: &#40;&#91;RS 725.116.21 - Ordonnance du 7 novembre 2007 concernant l&#x27;utilisation de l&#x27;impôt sur les huiles minérales à affectation obligatoire et des autres moyens affectés à la circulation routière... &#124; Fedlex&#93;&#40;https://www.fedlex.admin.ch/eli/cc/2007/797/fr&#35;art&#95;19&#41;&#41;&#91;LexFind&#93;&#40;https://www.lexfind.ch/fe/fr/tol/30723/fr&#41;Un troisième niveau existe, distinct des deux précédents : le périmètre du projet d&#x27;agglomération &#40;PA&#41;, délimité par l&#x27;organisme responsable et non par la Confédération. Pour Fribourg, le PA5 réunit 25 communes de la Sarine et de la Singine sur la base du périmètre VACo. La modalité « Fribourg &#40;Communes participantes PA&#41; » se réfère à ce périmètre-là.Sources<br>Espace à caractère urbain, agglomérations et typologies – Rapport explicatif &#40;OFS, n° 2298-2000&#41;: &#91;Actualisation 2020 des niveaux géographiques non-institutionnels - Espace à caractère urbain, agglomérations et typologies - Rapport explicatif &#124; Publication&#93;&#40;https://www.bfs.admin.ch/bfs/fr/home/statistiques/catalogues-banques-donnees.assetdetail.30665779.html&#41;<br>Programme en faveur du trafic d&#x27;agglomération de l&#x27;ARE : &#91;https://www.are.admin.ch/fr/projets-dagglomeration-de-6e-generation&#93;&#40;https://www.are.admin.ch/fr/projets-dagglomeration-de-6e-generation&#41;<br>Projet d&#x27;agglomération de 5e génération &#40;PA5&#41; – Fribourg : &#91;https://www.agglo-fr.ch/projets-dagglomeration/instrument-projet-dagglomeration/pa5&#93;&#40;https://www.agglo-fr.ch/projets-dagglomeration/instrument-projet-dagglomeration/pa5&#41;


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/gemeindetypologie)


## Metadata

- **identifier:** 22&#95;typologies@ssd
- **publisher.name:** ssd
- **organization.display&#95;name:** Amt für Statistik und Daten Freiburg
- **organization.url:** https://www.fr.ch/deef/ssd
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** typologie, commune, agglomeration, urbain, rural
- **issued:** 2026-03-13 00:00:00
- **metadata&#95;created:** 2026-02-12T01:02:56.147323
- **metadata&#95;modified:** 2026-10-14T03:07:31.180167
- **Resource id:** 39042c09-6936-4f74-b7f9-423c0747eb1f
- **Resource name:** csv
- **Resource package&#95;id:** 0d69e825-1a18-4e42-8a2a-2afe6e013d55
- **Resource description:** Typologies des communes &#40;csv&#41;
- **Resource issued:** 2026-03-13 00:00:00
- **Resource modified:** 2026-03-13 08:53:59.077000+00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;by
- **Resource rights:** http://www.opendefinition.org/licenses/cc-by-sa


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **csv** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://opendata.fr.ch/api/explore/v2.1/catalog/datasets/22_typologies/exports/csv?use_labels=true")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** statfr@fr.ch &#124; Centre de compétences et de coordination OGD / Kompetenz- und Koordinationszentrum für OGD
